# GANs (from a VAE learner’s perspective): slow, symbol-level recap + worked example

This notebook is a **detailed recap** of our discussion:
- What is $x_{\text{real}}$ vs $x_{\text{fake}}$?
- How $z$, $G$, and $D$ relate (and do **not** relate) to VAE parts
- The GAN objective and what it means when we **maximize** it
- Why training can **stall** if the discriminator becomes too good (saturation / vanishing gradients)
- A **worked toy example** you can run: a GAN that learns a 2D data distribution

> **Goal:** intuition → math → runnable code.

## 1) VAE vs GAN: what carries over, what changes

### VAE (Variational Autoencoder)
A standard VAE has:
- **Encoder** $q_\phi(z\mid x)$: maps data $x \to z$
- **Decoder** $p_\theta(x\mid z)$: maps latent $z \to x$

It is trained by maximizing the ELBO (Evidence Lower Bound):

$$\log p_\theta(x) \ge \mathbb{E}_{z\sim q_\phi(z\mid x)}[\log p_\theta(x\mid z)] - \mathrm{KL}(q_\phi(z\mid x)\,\|\,p(z)).$$

So VAEs explicitly use likelihood-style terms (via ELBO) and a KL regularizer.

---

### GAN (Generative Adversarial Network)
A basic GAN has:
- **Generator** $G_\theta(z)$: maps noise/latent $z \to x_{\text{fake}}$
- **Discriminator** $D_\psi(x)$: maps data $x \to (0,1)$ interpreted as “probability real”

Key differences from VAEs:
- Vanilla GAN has **no encoder** $x\to z$.
- It does **not** directly optimize $\log p(x)$.
- Training is a **two-player game** (minimax).

## 2) Are $x_{\text{real}}$ and $x_{\text{fake}}$ latent spaces?

**No.** They live in the **data space** (e.g., pixels for images).

- Real samples:

$$x_{\text{real}} \sim p_{\text{data}}(x).$$

- Fake samples:

$$z \sim p(z), \quad x_{\text{fake}} = G(z).$$

The latent/noise space is $z$ (often $z\sim \mathcal{N}(0,I)$).

## 3) Discriminator objective: what does maximizing mean?

When training the discriminator, a common objective is:

$$\mathcal{L}_D = \mathbb{E}_{x\sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z\sim p(z)}[\log(1 - D(G(z)))].$$

Interpretation term-by-term:

1) $\mathbb{E}[\log D(x)]$ pushes **$D(x_{\text{real}})\to 1$** (real should be labeled real).

2) $\mathbb{E}[\log(1 - D(G(z)))]$ pushes **$D(x_{\text{fake}})\to 0$** (fake should be labeled fake).

So $D$ is simply learning a binary classifier with cross-entropy / log-likelihood.

## 4) Generator objective: fooling the discriminator

The original minimax game is:

$$\min_G \max_D\; V(D,G) = \mathbb{E}_{x\sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z\sim p(z)}[\log(1 - D(G(z)))].$$

Two common generator losses:

**(A) Minimax / saturating (matches the minimax expression):**

$$\mathcal{L}^{\text{minimax}}_G = \mathbb{E}_{z\sim p(z)}[\log(1 - D(G(z)))],$$
and $G$ minimizes it.

**(B) Non-saturating (often used in practice):**

$$\mathcal{L}^{\text{ns}}_G = -\mathbb{E}_{z\sim p(z)}[\log D(G(z))],$$
and $G$ minimizes this (equivalently, maximizes $\log D(G(z))$).

## 5) Why training can stall if $D$ becomes too good (saturation)

Suppose discriminator becomes near-perfect early:

$$D(x_{\text{real}})\approx 1, \quad D(G(z))\approx 0.$$ 

Let $a = D(G(z))$.

### Minimax / saturating form
$$\mathcal{L}^{\text{minimax}}_G = \mathbb{E}[\log(1-a)].$$

Even if $\frac{d}{da}\log(1-a)$ is not tiny near $a\approx 0$, the gradient that reaches $G$ is:

$$\frac{d\mathcal{L}_G}{d\theta_G} = \frac{d\mathcal{L}_G}{da}\cdot \frac{da}{dG}\cdot \frac{dG}{d\theta_G}.$$ 

When $D$ uses a sigmoid output, it can **saturate** near 0 or 1, making $\frac{da}{dG}$ very small.
That produces **vanishing gradients** for $G$.

### Non-saturating form helps
$$\mathcal{L}^{\text{ns}}_G = -\mathbb{E}[\log(a)].$$

When $a\approx 0$, $-\log(a)$ is large and tends to provide a stronger learning signal in practice.

## 6) How GAN training is done in practice (alternating updates)

Repeat many times:

1. Sample a minibatch $\{x_i\}$ from real data.
2. Sample a minibatch $\{z_i\}$ from noise prior.
3. Make fakes $x_i^{\text{fake}} = G(z_i)$.
4. **Update $D$** to increase $\log D(x_{\text{real}})$ and $\log(1-D(x_{\text{fake}}))$.
5. Sample new $\{z_i\}$.
6. **Update $G$** to increase $D(G(z))$ (using a generator loss).

We typically do **small updates** to $D$ and $G$ alternately so neither gets too far ahead.

## 7) After training, how do we use the GAN?

Usually we **discard $D$** and keep **only $G$**.

To generate new samples:

$$z\sim p(z) \quad\Rightarrow\quad x = G(z).$$

This is the generative part—similar to sampling from a VAE prior and decoding, but trained via adversarial feedback.

---

# Worked example: a tiny GAN learning a 2D distribution (PyTorch)

We train on a mixture of Gaussians arranged in a circle (multiple modes). This is fast and lets you visually see $G$ improve.

In [ ]:

# If you run this notebook locally and PyTorch isn't installed, uncomment:
# !pip install torch

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)
np.random.seed(0)


## 7.1 Real data sampler

We sample from $k$ Gaussian clusters on a circle. This produces multi-modal data.


In [ ]:

def sample_real(batch_size: int, radius: float = 2.0, noise: float = 0.15, k: int = 8):
    angles = np.random.randint(0, k, size=batch_size) * (2*np.pi / k)
    centers = np.stack([radius*np.cos(angles), radius*np.sin(angles)], axis=1)
    x = centers + noise*np.random.randn(batch_size, 2)
    return x.astype(np.float32)

x = sample_real(2000)
plt.figure()
plt.scatter(x[:,0], x[:,1], s=5)
plt.title("Real data samples (2D mixture on a circle)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()


## 7.2 Define $G$ and $D$

- $G: \mathbb{R}^2\to\mathbb{R}^2$
- $D: \mathbb{R}^2\to(0,1)$ (sigmoid output)


In [ ]:

class Generator(nn.Module):
    def __init__(self, z_dim=2, hidden=64, x_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, x_dim),
        )
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, x_dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

G = Generator()
D = Discriminator()


## 7.3 Loss functions

Discriminator loss (minimize negative of the maximization objective):

$$\ell_D = -\Big(\mathbb{E}[\log D(x_{\text{real}})] + \mathbb{E}[\log(1 - D(x_{\text{fake}}))]\Big).$$

Generator loss (non-saturating):

$$\ell_G = -\mathbb{E}[\log D(G(z))].$$


In [ ]:

def d_loss_fn(d_real, d_fake, eps=1e-8):
    return -(torch.log(d_real + eps).mean() + torch.log(1 - d_fake + eps).mean())

def g_loss_fn(d_fake, eps=1e-8):
    return -torch.log(d_fake + eps).mean()


## 7.4 Training loop (alternating updates)

We alternate updates: train $D$ a little, then train $G$ a little.


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
G.to(device); D.to(device)

optD = optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.9))
optG = optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.9))

def sample_z(batch_size: int, z_dim: int = 2):
    return torch.randn(batch_size, z_dim, device=device)

@torch.no_grad()
def plot_samples(step, n=2000):
    z = sample_z(n)
    fake = G(z).cpu().numpy()
    real = sample_real(n)
    plt.figure()
    plt.scatter(real[:,0], real[:,1], s=5, label="real")
    plt.scatter(fake[:,0], fake[:,1], s=5, label="fake")
    plt.title(f"Real vs Fake samples (step={step})")
    plt.xlabel("x1"); plt.ylabel("x2")
    plt.legend()
    plt.show()

steps = 3000
batch_size = 256
d_steps = 1

d_losses, g_losses = [], []

for step in range(1, steps+1):
    # ---- Train D ----
    for _ in range(d_steps):
        x_real = torch.tensor(sample_real(batch_size), device=device)
        z = sample_z(batch_size)
        x_fake = G(z).detach()

        d_real = D(x_real)
        d_fake = D(x_fake)

        lossD = d_loss_fn(d_real, d_fake)

        optD.zero_grad()
        lossD.backward()
        optD.step()

    # ---- Train G ----
    z = sample_z(batch_size)
    x_fake = G(z)
    d_fake = D(x_fake)

    lossG = g_loss_fn(d_fake)

    optG.zero_grad()
    lossG.backward()
    optG.step()

    d_losses.append(lossD.item())
    g_losses.append(lossG.item())

    if step in [1, 200, 500, 1000, 2000, 3000]:
        plot_samples(step)


## 7.5 Plot losses

GAN losses are game dynamics, so they may not be monotonic.


In [ ]:

plt.figure()
plt.plot(d_losses, label="D loss")
plt.plot(g_losses, label="G loss")
plt.title("Training losses")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()


## 8) Quick numeric intuition: why non-saturating can help when $a=D(G(z))$ is tiny

Compare gradients w.r.t. $a$:

- Minimizing $\log(1-a)$:
$$\frac{d}{da}\log(1-a) = -\frac{1}{1-a}.$$

- Minimizing $-\log(a)$:
$$\frac{d}{da}(-\log a) = -\frac{1}{a}.$$

If $a$ is very small, $-1/a$ has large magnitude, often yielding stronger updates.


In [ ]:

def grad_minimax(a):
    return -1.0/(1.0-a)

def grad_nonsat(a):
    return -1.0/a

for a in [1e-6, 1e-4, 1e-2, 0.1, 0.5, 0.9]:
    print(f"a={a:>8} | d/da log(1-a)={grad_minimax(a):>10.3e} | d/da(-log a)={grad_nonsat(a):>10.3e}")


## 9) Final recap checklist

- $x_{\text{real}}$: real dataset sample (data space)
- $x_{\text{fake}} = G(z)$: generated sample (data space)
- $z$: latent/noise input (often $\mathcal{N}(0,I)$)
- $G$: decoder-like mapping $z\to x$
- $D$: classifier mapping $x\to(0,1)$ (not a VAE encoder)
- Training alternates updates for $D$ and $G$
- If $D$ saturates too early, gradients to $G$ can vanish
- After training, keep $G$ to generate new samples
